# Forest biomass from a single Sentinel-2 image

Every figure and number in the report comes from this notebook, which only imports from `biomass/`. Run `make evaluate` first.

In [1]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from biomass import CACHE, FIGURES, MODELS, RAW
from biomass.carbon import tco2_per_patch
from biomass.evaluate import MODEL_NAMES, metrics, predict, residual_plot, residuals_by_bin, scatter
from biomass.features import load_features, ndvi
from biomass.load import BANDS, load_patches, read_label, read_s2
from biomass.models import inputs
from biomass.split import load_split
FIGURES.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10})

## Dataset

In [2]:
p, split, feats = load_patches(), load_split(), load_features()
selection = pd.read_csv(CACHE / "selection.csv").set_index("chip_id")
print(f"{len(p['ids'])} chips, patches {p['X'].shape[1]}×{p['X'].shape[2]}×{p['X'].shape[3]}")
print({k: len(v) for k, v in split.items()})
print(selection.month.value_counts().to_dict())

200 chips, patches 64×64×10
{'train': 140, 'val': 30, 'test': 30}
{'July': 72, 'August': 71, 'June': 57}


In [3]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(p["y"], bins=30, color="#2f5d3a")
ax.set(xlabel="Patch-mean above-ground biomass (t/ha)", ylabel="Chips")
fig.tight_layout(); fig.savefig(FIGURES / "biomass_hist.png"); plt.show()

/var/folders/w_/x7zjh85x6z3f4gpz8b28yx3w0000gn/T/ipykernel_15959/1293017965.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGURES / "biomass_hist.png"); plt.show()


## Three example chips: true colour, NDVI, LiDAR biomass

In [4]:
examples = [p["ids"][i] for i in np.argsort(p["y"])[[len(p["y"]) // 10, len(p["y"]) // 2, -len(p["y"]) // 10]]]
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
rgb = [BANDS.index(b) for b in ("B4", "B3", "B2")]
for row, chip in zip(axes, examples):
    refl, _ = read_s2(RAW / selection.filename[chip]); label = read_label(RAW / selection.label[chip])
    row[0].imshow(np.clip(refl[..., rgb] / 0.25, 0, 1)); row[0].set_title(f"{chip}: true colour")
    im1 = row[1].imshow(ndvi(refl), vmin=0, vmax=0.9, cmap="YlGn"); row[1].set_title("NDVI")
    im2 = row[2].imshow(label, vmin=0, vmax=250, cmap="viridis"); row[2].set_title(f"LiDAR biomass, mean {label.mean():.0f} t/ha")
    for ax in row: ax.set_axis_off()
fig.colorbar(im1, ax=axes[:, 1], shrink=0.5, label="NDVI"); fig.colorbar(im2, ax=axes[:, 2], shrink=0.5, label="t/ha")
fig.savefig(FIGURES / "example_chips.png", bbox_inches="tight"); plt.show()

/var/folders/w_/x7zjh85x6z3f4gpz8b28yx3w0000gn/T/ipykernel_15959/204574600.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(FIGURES / "example_chips.png", bbox_inches="tight"); plt.show()


## NDVI against biomass

In [5]:
y_by_id = pd.Series(p["y"], index=p["ids"])
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(feats.ndvi, y_by_id[feats.index], s=8, alpha=0.6, color="#2f5d3a")
ax.set(xlabel="Mean NDVI", ylabel="LiDAR biomass (t/ha)")
print(f"Pearson r = {np.corrcoef(feats.ndvi, y_by_id[feats.index])[0, 1]:.2f}")
fig.tight_layout(); fig.savefig(FIGURES / "ndvi_vs_biomass.png"); plt.show()

Pearson r = 0.33


/var/folders/w_/x7zjh85x6z3f4gpz8b28yx3w0000gn/T/ipykernel_15959/2892763836.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGURES / "ndvi_vs_biomass.png"); plt.show()


## Model comparison on the test set

In [6]:
test = inputs(split)["test"]
names = [n for n in MODEL_NAMES if (MODELS / f"{n}_settings.json").exists()]
preds = {n: predict(n, test) for n in names}
table = pd.DataFrame({MODEL_NAMES[n]: metrics(test["y"], pr) for n, pr in preds.items()}).T
table.round(2)

,mae,rmse,r2,rmse_tco2_per_patch,mae_tco2_per_patch
Ridge,11.29,13.48,0.62,15222.82,12753.54


In [7]:
scatter(test["y"], preds, FIGURES / "scatter.png"); residual_plot(test["y"], preds, FIGURES / "residuals.png")
pd.concat({MODEL_NAMES[n]: residuals_by_bin(test["y"], pr) for n, pr in preds.items()}, axis=1).round(1)

Ridge                    
            n mean_residual   mae
bin                              
0–50        7           2.8  12.1
50–100     23           0.4  11.1
100–150     0           0.0   0.0
150+        0           0.0   0.0

Optical reflectance saturates as the canopy closes, so every model under-predicts the densest patches: the residual is most negative in the highest biomass bin.

## Error in tonnes of CO₂ per patch

Above-ground only, carbon fraction 0.47, patch area 655.36 ha.

In [8]:
pd.DataFrame({MODEL_NAMES[n]: {"RMSE (t/ha)": metrics(test["y"], pr)["rmse"],
                              "RMSE (tCO2 / patch)": tco2_per_patch(metrics(test["y"], pr)["rmse"]),
                              "mean true stock (tCO2 / patch)": tco2_per_patch(float(test["y"].mean()))}
              for n, pr in preds.items()}).T.round(0)

,RMSE (t/ha),RMSE (tCO2 / patch),mean true stock (tCO2 / patch)
Ridge,13.0,15223.0,71832.0
